In [1]:
from pathlib import Path
import os

In [2]:
from main_train import get_args
from utils.config import load_config
from data.dataset import get_vqav2
from data.dataset import get_filtered_trainval
from data.text_processing import save_tokenizer

from data.custom_generators import get_custom_generators

2025-08-06 09:28:59.675021: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1754465339.697246    7708 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1754465339.703760    7708 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2025-08-06 09:28:59.726878: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [3]:
config = load_config()
config['model']

{'max_length': 15,
 'num_vocab_words': -1,
 'min_frequency': 5,
 'image_size': 224,
 'num_channels': 3,
 'num_classes': 1000,
 'consider_teacher': True,
 'k_window': 5,
 'output_MFB': 1024,
 'num_attention_glimps': 2,
 'embedding_dim': 100,
 'use_glove': True,
 'dropout_rate': 0}

In [4]:
config['training']

{'num_epochs': 10,
 'lr': 0.0001,
 'batch_size': 32,
 'alpha': 0.1,
 'temperature': 3}

In [5]:
from datetime import datetime

In [6]:
now = datetime.now().strftime("%y%m%d_%H%M")
saving_folder = config["paths"]["output_path"] / f"{'MFBBaseline'}_{now}"
saving_folder.mkdir(parents=True, exist_ok=True)

config["paths"]["saving_folder"] = saving_folder
config["model"]["model_architecture"] = "MFBBaseline"
# config["training"]["knowledge_distillation"] = args.distill
config["training"]["knowledge_distillation"] = True

In [7]:
saving_folder

PosixPath('outputs/MFBBaseline_250806_0929')

In [8]:
if config["model"]["num_vocab_words"] > 0:
    config["model"]["min_frequency"] = 0
    num_words = config["model"]["num_vocab_words"]
    tokenizer_path = config["paths"]["output_path"] / f"word_index{num_words}.json"
else:
    mf = config["model"]["min_frequency"]
    tokenizer_path = config["paths"]["output_path"] / f"word_index_mf{mf}.json"
if not tokenizer_path.is_file():
    #save_tokenizer(config, tokenizer_path, verbose=False)
    pass
tokenizer_path

PosixPath('outputs/word_index_mf5.json')

In [9]:
from models.vqa_models import get_model

In [10]:
config['model']["num_vocab_words"] = 6415

In [11]:
model = get_model(config, tokenizer_path)

I0000 00:00:1754465344.951240    7708 gpu_device.cc:2022] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 5180 MB memory:  -> device: 0, name: NVIDIA GeForce GTX 1060 6GB, pci bus id: 0000:21:00.0, compute capability: 6.1


Number of parameters: 24419360


In [12]:
config["model"]["model_architecture"] = "MFBAttention"
config["model"]["num_attention_glimps"] = 2
model = get_model(config, tokenizer_path)

Number of parameters: 40421154


In [13]:
config["model"]["model_architecture"] = "MFBCoAttention"
config["model"]["num_attention_glimps"] = 2
model = get_model(config, tokenizer_path)

Number of parameters: 51169828
